# Hex

Donea Fernando-Emanuel

grupa 243

In [6]:
import heapq
from copy import deepcopy

MAX=1 #jucator ROSU -> sus pana jos
MIN=2 # jucator ALBASTRU -> stanga dreapta
GOL=0
WIN=10000000
INFINIT=10000000000000000000
N=11


class Nod:
    #as fi numit-o mai degraba stare ca e mai "fitting" dar asa am facut pana acum asa ramane
    def __init__(self, info, parinte=None, detalii_mutare=None):
        self.parinte = parinte
        self.info = info
        self.detalii=detalii_mutare

    def vecini(self, i, j):

        #vector de coord
        directii=[(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0)]#sus sus-dreapta stanga dreapta jos-stanga jos
        lista_vecini=[]

        for x,y in directii:
            i_nou=i+x
            j_nou=j+y
            if 0<=i_nou<N and 0<=j_nou<N:
                lista_vecini.append((i_nou,j_nou))
        return lista_vecini


    #facem dijkstra pentru a afla distanta minima
    #calculam distanta de la start la finish
    def distanta_dijkstra(self, jucator):
        distante={}
        pq=[]

        if jucator==MAX: #max cauta sa uneasca linia 0 cu linia n-1

            for j in range(N): #puneam in pq toata linia 0 (toate starturile posibile)
                if self.info[0][j] !=MIN: #daca slotul nu e al adversarului
                    if self.info[0][j]==MAX:
                        cost=0
                    else:
                        cost=1

                    heapq.heappush(pq,(cost, 0, j))
                    distante[(0,j)]=cost

        else: #min catua sa uneasca coloana 0 cu coloana n-1
            for i in range(N):
                if self.info[i][0] !=MAX:
                    if self.info[i][0]==MIN:
                        cost=0
                    else:
                        cost=1
                    heapq.heappush(pq,(cost, i, 0))
                    distante[(i,0)]=cost

        while pq:
            cost_curent, i,j= heapq.heappop(pq)

            if jucator==MAX and i==N-1:
                return cost_curent
            if jucator==MIN and j==N-1:
                return cost_curent

            #daca costul curent e mai mare decat costul stocat in dictionar,
            # inseamna ca am ajuns la un nod care a fost vizitat deja cu un cost mai mic deci ignoram
            if cost_curent > distante.get((i,j), INFINIT):
                continue

            #parcurgem mutarile posbile
            for i_vecin, j_vecin in self.vecini(i,j):
                if jucator==MAX:
                    adversar=MIN
                else:
                    adversar=MAX

                #daca vecinul nu e al adversarului, adica gol sau al meu pot sa merg mai departe
                if self.info[i_vecin][j_vecin]!=adversar:

                    if self.info[i_vecin][j_vecin]==jucator:
                        cost_pas=0
                    else:
                        cost_pas=1
                    cost_nou=cost_pas+cost_curent

                    if cost_nou < distante.get((i_vecin,j_vecin), INFINIT):
                        distante[(i_vecin,j_vecin)]=cost_nou
                        heapq.heappush(pq,(cost_nou, i_vecin, j_vecin))

        return INFINIT


    def evaluare(self):
         #aici implementam functia de evaluare

        dist_maxi=self.distanta_dijkstra(MAX) #cate mutari mai are nevoie MAX pentru a castiga
        dist_mini=self.distanta_dijkstra(MIN)

        if dist_maxi==0:
            return WIN
        if dist_mini==0:
            return -WIN

        if dist_maxi==INFINIT:
            return -WIN +1000
        if dist_mini==INFINIT:
            return WIN-1000

        return dist_mini-dist_maxi


    def __str__(self):
        rezultat=""
        for i in range(N):
            rezultat+=" "*i
            for j in range(N):
                rezultat+=str(self.info[i][j])+ " "
            rezultat+="\n"
        return rezultat



    def __repr__(self):
        return self.__str__()



In [7]:
class Tree:
    def __init__(self, nod_start=None):
        self.n=11
        if nod_start is not None:
            self.nod_start=nod_start
        else:
            self.nod_start=Nod([[GOL for _ in range(self.n)] for _ in range(self.n)],parinte=None, detalii_mutare=None)




    def este_final(self, nod): #true daca e stare finala. eu personal as returna si castigatorul/draw (intre 0,1,2,3 de ex, 0 draw, 1 castiga AI 2 castiga om, 3 nu e final). va ia din munca mai incolo

        dist_max=nod.distanta_dijkstra(MAX)
        dist_min=nod.distanta_dijkstra(MIN)

        if dist_max==0:
            return True, MAX
        if dist_min==0:
            return True,MIN

        #verificam daca e remiza
        for i in range(11):
            for j in range(11):
                if nod.info[i][j]==GOL:
                    return False, None

        return True, 0




    def succesori(self, nod, jucator):
#fill in the blanks unde e mai sus, posibil sa nu aveti nevoie si de str si de repr. atata timp cat se afiseaza lizibil starile prin care trecem e ok
        lista_succesori=[]
        for i in range(11):
            for j in range(11):
                if nod.info[i][j]==GOL:

                    matrice_noua=deepcopy(nod.info)
                    matrice_noua[i][j]=jucator
                    lista_succesori.append(Nod(matrice_noua, parinte=nod, detalii_mutare=(i,j)))

        return lista_succesori





**Justificare functie de evaluare**

Functia de evaluare pentru algoritmul MinMax estimeaza cat de avantajoasa este o stare pentru jucatorul curent. O valoare pozitiva mare indica un avantaj clar pentru jucatorul MAX (rosu), iar o valoare negativa indica un avantaj pentru MIN (albastru).

Functia propusa calculeaza **diferenta** dintre distanta _minima ramasa a jucatorului curent_ si _distanta minima a adversarului_:

$evaluare(stare)= distJucatorMIN - distantaJucatorMAX $

Pentru a determina aceaste distante folosim Dijkstra direct pe table de 11x11. Cum scopul jocului este crearea unui pod cotinuu intre cele doua margini opuse, costurile pe drum sunt setate astfel:
- `cost=0` pe un slot care are deja piesa mea (adica e parte din drum)
- `cost=1` pe un slot gol (e nevoie o mutare)
- `cost=inifit` pe un slot ocupar de un adversar (drumul e blocat)

Astfel, `dist_maxi` este fix numarul minim de piese de care MAX mai are nevoie pentru a castiga. MAX vrea sa si faca drumul cat mai scurt, dar vrea sa il si blocheze pe MIN, lungidu-i drumul. Diferenta dintre `dist_maxi` si `dist_mini` creste proportional cu avantajul de pe tabla.

Daca o distanta ajunge la 0, inseamna ca jucatorul a unit marginile si a castigat. Daca distanta ajunge la infinit, jucatorul a fost blocat complet de adversar.

In [8]:
def alpha_beta(nod, adancime, jucator, tree, alpha, beta):

    final,castigator=tree.este_final(nod=nod)

    if adancime==0 or final:
        return nod.evaluare(), nod

    if jucator==MAX:
        maxi=-WIN*5
        maxiNod = None
        succesori=tree.succesori(nod, jucator=MAX)

        for succesor in succesori:
            evaluare,_ =alpha_beta(succesor, adancime-1, MIN, tree, alpha, beta)
            if evaluare>maxi:
                maxi=evaluare
                maxiNod=succesor

            alpha = max(alpha, maxi)
            if beta <= alpha:
                break  #taiem ramura

        return maxi, maxiNod

    else:
        mini=5*WIN
        miniNod=None
        succesori=tree.succesori(nod, jucator=MIN)

        for succesor in succesori:
            evaluare,_=alpha_beta(succesor, adancime-1, MAX, tree, alpha, beta)
            if evaluare<mini:
                mini=evaluare
                miniNod=succesor

            beta = min(beta, mini)
            if beta <= alpha:
                break  #taiem ramura

        return mini, miniNod

In [9]:
def ai_vs_om():
    joc=Tree()
    stare_curenta=joc.nod_start
    rand_curent=MAX #om

    while True:
        print("Tabla curenta")
        print(stare_curenta)

        final,castigator=joc.este_final(stare_curenta)

        if final is not False:
            if castigator==0:
                print("Egalitate")
            else:
                print(f"A castigat {castigator}")
            break


        if rand_curent==MAX:
            print("E randul tau. Introdu mutarea: i=linie, j=coloana")
            i=int(input("Linie:"))
            j=int(input("Coloana: "))

            if 0<=i<N and 0<=j<=N:
                if stare_curenta.info[i][j]==GOL:
                    #facem mutarea
                    matrice_noua=deepcopy(stare_curenta.info)
                    matrice_noua[i][j]=MAX
                    stare_curenta=Nod(matrice_noua, parinte=stare_curenta, detalii_mutare=(i,j))
                    rand_curent=MIN #dam switch randul la AI
                else:
                    print("Pozitia ocupata")
            else:
                print("pozitie incorecta")
        else:
            print("AI se gandeste")
            scor, mutare=alpha_beta(stare_curenta, adancime=2, jucator=MIN, tree=joc, alpha=-INFINIT, beta=INFINIT)
            if mutare:
                print(f"Ai-ul a mutat pe {mutare.detalii} ")
                stare_curenta=mutare

            rand_curent=MAX


def ai_vs_ai():
    joc=Tree()
    stare_curenta=joc.nod_start
    rand_curent=MAX #om

    while True:
        print("Tabla curenta")
        print(stare_curenta)

        final,castigator=joc.este_final(stare_curenta)

        if final is not False:
            if castigator==0:
                print("Egalitate")
            else:
                print(f"A castigat {castigator}")
            break


        if rand_curent==MAX:
            scor, mutare=alpha_beta(stare_curenta, adancime=2, jucator=MAX, tree=joc, alpha=-INFINIT, beta=INFINIT)
            if mutare:
                print(f"Ai-ul:MAX a mutat pe {mutare.detalii} ")
                stare_curenta=mutare
            rand_curent=MIN

        else:
            scor, mutare=alpha_beta(stare_curenta, adancime=2, jucator=MIN, tree=joc, alpha=-INFINIT, beta=INFINIT)
            if mutare:
                print(f"Ai-ul:MIN a mutat pe {mutare.detalii} ")
                stare_curenta=mutare

            rand_curent=MAX




In [10]:
ai_vs_ai()

Tabla curenta
0 0 0 0 0 0 0 0 0 0 0 
 0 0 0 0 0 0 0 0 0 0 0 
  0 0 0 0 0 0 0 0 0 0 0 
   0 0 0 0 0 0 0 0 0 0 0 
    0 0 0 0 0 0 0 0 0 0 0 
     0 0 0 0 0 0 0 0 0 0 0 
      0 0 0 0 0 0 0 0 0 0 0 
       0 0 0 0 0 0 0 0 0 0 0 
        0 0 0 0 0 0 0 0 0 0 0 
         0 0 0 0 0 0 0 0 0 0 0 
          0 0 0 0 0 0 0 0 0 0 0 

Ai-ul:MAX a mutat pe (0, 1) 
Tabla curenta
0 1 0 0 0 0 0 0 0 0 0 
 0 0 0 0 0 0 0 0 0 0 0 
  0 0 0 0 0 0 0 0 0 0 0 
   0 0 0 0 0 0 0 0 0 0 0 
    0 0 0 0 0 0 0 0 0 0 0 
     0 0 0 0 0 0 0 0 0 0 0 
      0 0 0 0 0 0 0 0 0 0 0 
       0 0 0 0 0 0 0 0 0 0 0 
        0 0 0 0 0 0 0 0 0 0 0 
         0 0 0 0 0 0 0 0 0 0 0 
          0 0 0 0 0 0 0 0 0 0 0 

Ai-ul:MIN a mutat pe (0, 3) 
Tabla curenta
0 1 0 2 0 0 0 0 0 0 0 
 0 0 0 0 0 0 0 0 0 0 0 
  0 0 0 0 0 0 0 0 0 0 0 
   0 0 0 0 0 0 0 0 0 0 0 
    0 0 0 0 0 0 0 0 0 0 0 
     0 0 0 0 0 0 0 0 0 0 0 
      0 0 0 0 0 0 0 0 0 0 0 
       0 0 0 0 0 0 0 0 0 0 0 
        0 0 0 0 0 0 0 0 0 0 0 
         0 0 0 0 0 0 0 0 0 0 0 
       